In [1]:
import uuid
import chromadb #轻量级的向量数据库
import requests
import os
from openai import OpenAI

In [2]:
#创建一个向量数据库
client = chromadb.PersistentClient(path="./db/chroma_demo") #向量数据库当做是一个磁盘文件夹
#基于向量数据库创建存储向量的数据集合
collection = client.get_or_create_collection(name="collection_v2") #数据集合当做是一个磁盘文件

In [3]:
#构建一个数据切分的函数（原始知识库文件内容进行分块处理）
def file_chunk_list(filePath):
    #1.进行原始文件数据的读取
    with open(filePath,'r',encoding='utf-8') as fp:
        data = fp.read()
    #2.进行数据切分
    chunk_list = data.split('\n\n')
    return chunk_list

In [8]:
#构建一个向量转换的函数实现（阿里百炼向量模型）
def embedding_by_api(text):
    #1.创建向量模型客户端
    client = OpenAI(
        api_key="sk-52xxxx6712",  # 如果您没有配置环境变量，请在此处用您的API Key进行替换
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"  # 百炼服务的base_url
    )
    #2.向量模型调用
    completion = client.embeddings.create(
        model="text-embedding-v3",
        input=text,
        dimensions=1024,
        encoding_format="float"
    )
    return completion.data[0].embedding

In [10]:
#构建一个模型调用的函数
def generate_by_api(prompt):
    ds_api_key = "sk-4b79xxxx66ebb425b3"
    # 实例化客户端
    client = OpenAI(api_key=ds_api_key, 
                    base_url="https://api.deepseek.com")

    # 调用 deepseek-r1 模型
    response = client.chat.completions.create(
        model="deepseek-reasoner", #调用推理模型deepseek-r1 模型标识/名称，存在推理过程
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    # 最终回复
    return response.choices[0].message.content

In [19]:
#代码的集成
#1.数据分块处理
documents = file_chunk_list("./中医问诊.txt")

#2.将分块结果进行向量转换
embeddings = [] #存放对每一个分块结果进行向量转换后的结果
for doc in documents:
    doc_emd = embedding_by_api(doc)
    embeddings.append(doc_emd)
    
#3.将生成的向量结果添加到向量数据库
ids = []
for _ in documents:
    ids.append(str(uuid.uuid4()))
    
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)

#4.接受用户提问，去向量数据库中进行检索
qs = '我好像是感冒了，症状是头痛、轻微发烧、肢节酸痛、打喷嚏和流鼻涕。'
qs_embedding = embedding_by_api(qs)#将用户提问转换成向量表示
#去向量数据库中检索和用户提问向量相似度最高的1个结果
res = collection.query(query_embeddings=[qs_embedding,],query_texts=qs,n_results=1)
result = res['documents'][0]
context = '\n'.join(result)
                    
#5.进行提示词构建
prompt = f'''你是一个中医问答机器人，任务是根据参考信息回答用户问题，如果你参考信息不足以回答用户问题，请回复不知道，切记不要去杜撰和自由发挥任何内容和信息，请用中文回答，参考信息：{context},来回答问题:{qs},'''

#6.进行模型调用
result = generate_by_api(prompt)
print(result)

根据您提供的症状（头痛、轻微发烧、肢节酸痛、打喷嚏和流鼻涕），这些部分匹配参考信息中风寒感冒的描述（如头痛、发热轻、肢节酸痛、鼻痒喷嚏、时流清涕）。但参考信息中风寒感冒的关键症状还包括恶寒重、无汗等，您未提及这些内容。因此，参考信息不足以完全确认您的感冒类型，我无法给出具体诊断或药方建议。

如果您有更多症状细节（如是否恶寒、出汗情况、鼻涕颜色等），建议咨询专业中医师进行详细诊断。
